In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(
    "u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [2]:
print(ratings.shape)
print(ratings.info())

print("Number of users:", ratings["user_id"].nunique())
print("Number of movies:", ratings["movie_id"].nunique())

print(ratings["rating"].value_counts().sort_index())

(100000, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
None
Number of users: 943
Number of movies: 1682
rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [3]:
print(ratings.shape)
print(ratings.info())

print("Number of users:", ratings["user_id"].nunique())
print("Number of movies:", ratings["movie_id"].nunique())

print(ratings["rating"].value_counts().sort_index())

(100000, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
None
Number of users: 943
Number of movies: 1682
rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [4]:
user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
user_item_filled = user_item_matrix.fillna(0)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


In [7]:
def get_similar_users(user_id, n=10):

    similarities = user_similarity_df[user_id]

    similar_users = (
        similarities
        .drop(user_id)
        .sort_values(ascending=False)
        .head(n)
    )

    return similar_users

In [8]:
get_similar_users(1, 10)

,1
user_id,
916,0.569066
864,0.547548
268,0.542077
92,0.540534
435,0.538665
457,0.538476
738,0.527031
429,0.525950
303,0.525718


In [9]:
def recommend_movies(user_id, n_recommendations=10, n_neighbors=20):

    similar_users = get_similar_users(
        user_id,
        n=n_neighbors
    )

    neighbor_ids = similar_users.index

    neighbor_ratings = user_item_matrix.loc[neighbor_ids]

    predicted_ratings = neighbor_ratings.mean(axis=0)

    watched_movies = user_item_matrix.loc[user_id].dropna().index

    predicted_ratings = predicted_ratings.drop(
        watched_movies,
        errors="ignore"
    )

    recommendations = predicted_ratings.sort_values(
        ascending=False
    ).head(n_recommendations)

    return recommendations

In [10]:
recommend_movies(1)

,0
movie_id,
602,5.0
736,5.0
492,5.0
853,5.0
1589,5.0
1168,5.0
1007,5.0
1143,5.0
1009,5.0


In [11]:
def recommend_movies_weighted(user_id, n_recommendations=10, n_neighbors=20):

    similar_users = get_similar_users(
        user_id,
        n=n_neighbors
    )

    neighbor_ids = similar_users.index

    neighbor_ratings = user_item_matrix.loc[neighbor_ids]

    weights = similar_users.values

    weighted_ratings = neighbor_ratings.mul(weights, axis=0)

    predicted_ratings = weighted_ratings.sum(axis=0) / (
        neighbor_ratings.notna().mul(weights, axis=0).sum(axis=0)
    )

    watched_movies = user_item_matrix.loc[user_id].dropna().index

    predicted_ratings = predicted_ratings.drop(
        watched_movies,
        errors="ignore"
    )

    return predicted_ratings.sort_values(
        ascending=False
    ).head(n_recommendations)

In [12]:
recommend_movies_weighted(1)

,0
movie_id,
1007,5.0
602,5.0
492,5.0
285,5.0
1168,5.0
1589,5.0
641,5.0
736,5.0
853,5.0


In [14]:
movies = pd.read_csv(
    "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [15]:
def show_recommendations(user_id, n=10):

    recs = recommend_movies_weighted(
        user_id,
        n_recommendations=n
    )

    result = recs.reset_index()
    result.columns = ["movie_id", "predicted_rating"]

    result = result.merge(
        movies,
        on="movie_id"
    )

    return result[
        ["movie_id", "title", "predicted_rating"]
    ]

In [16]:
show_recommendations(1)

,movie_id,title,predicted_rating
0,1007,Waiting for Guffman (1996),5.0
1,602,"American in Paris, An (1951)",5.0
2,492,East of Eden (1955),5.0
3,285,Secrets & Lies (1996),5.0
4,1168,Little Buddha (1993),5.0
5,1589,Schizopolis (1996),5.0
6,641,Paths of Glory (1957),5.0
7,736,Shadowlands (1993),5.0
8,853,Braindead (1992),5.0
9,1143,Hard Eight (1996),5.0


In [17]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

In [19]:
train_matrix = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
).fillna(0)

In [20]:
train_similarity = cosine_similarity(train_matrix)

train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

In [21]:
def precision_at_k(recommended_movies, relevant_movies, k=10):

    recommended_k = recommended_movies[:k]

    relevant_recommended = set(recommended_k) & set(relevant_movies)

    return len(relevant_recommended) / k

In [22]:
recommended = [10, 20, 30, 40, 50]

relevant = [10, 30, 50, 80]

print(precision_at_k(recommended, relevant, k=5))

0.6


In [23]:
item_similarity = cosine_similarity(
    user_item_filled.T
)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

In [24]:
def similar_movies(movie_id, n=10):

    return (
        item_similarity_df[movie_id]
        .drop(movie_id)
        .sort_values(ascending=False)
        .head(n)
    )

In [25]:
similar_movies(50)

,50
movie_id,
181,0.884476
174,0.764885
172,0.749819
1,0.734572
127,0.697332
121,0.692837
210,0.689343
100,0.686533
98,0.676428


In [26]:
from sklearn.decomposition import TruncatedSVD

matrix = user_item_filled.values

svd = TruncatedSVD(
    n_components=20,
    random_state=42
)

user_features = svd.fit_transform(matrix)

print(user_features.shape)

(943, 20)
